In [1]:
import pandas as pd
import zipfile_deflate64 as zipfile
import numpy as np

In [2]:
# Carga de datos
zf = zipfile.ZipFile("../BlackjackData/blackjack_dataset.zip") 
df_raw = pd.read_csv(zf.open("blackjack_dataset.csv"))

In [3]:
df = df_raw[~df_raw['actions_taken'].str.contains('\\[]')].sample(frac=0.01) # Cogemos una fracción de los datos porque son 50.000.000 en total
df.head(15)

,shoe_id,cards_remaining,dealer_up,initial_hand,dealer_final,dealer_final_value,player_final,player_final_value,actions_taken,run_count,true_count,win
45560052,749782,79,4,"[8, 10]","[4, 9, 10]",23,"[[8, 10]]",[18],[['S']],-6,-3,1.0
24420850,401890,281,9,"[10, 9]","[9, 10]",19,"[[10, 9]]",[19],[['S']],-11,-2,0.0
35369487,582076,398,9,"[10, 11]","[9, 7, 2]",18,"[[10, 11]]",['BJ'],[['S']],2,0,1.5
242040,3982,299,9,"[7, 8]","[9, 11]",20,"[[7, 8, 10]]",[25],[['H']],12,2,-1.0
21796863,358698,259,8,"[10, 4]","[8, 4, 10]",22,"[[10, 4, 9]]",[23],[['H']],2,0,-1.0
40871723,672630,401,11,"[4, 10]","[11, 4, 9, 4]",18,"[[4, 10, 10]]",[24],"[['N', 'H']]",-1,0,-1.0
40809795,671609,189,10,"[7, 3]","[10, 6, 2]",18,"[[7, 3, 8]]",[18],"[['H', 'S']]",2,0,0.0
6453934,106205,320,10,"[5, 7]","[10, 5, 10]",25,"[[5, 7, 10]]",[22],[['H']],12,1,-1.0
8092807,133175,356,6,"[7, 7]","[6, 11, 4]",21,"[[7, 4, 10], [7, 10], [7, 5]]","[21, 17, 12]","[['P', 'D'], ['P', 'S'], ['S']]",-1,0,-2.0
25066232,412509,138,4,"[2, 10]","[4, 10, 10]",24,"[[2, 10]]",[12],[['S']],-10,-3,1.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 485409 entries, 45560052 to 12898836
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   shoe_id             485409 non-null  int64  
 1   cards_remaining     485409 non-null  int64  
 2   dealer_up           485409 non-null  int64  
 3   initial_hand        485409 non-null  object 
 4   dealer_final        485409 non-null  object 
 5   dealer_final_value  485409 non-null  object 
 6   player_final        485409 non-null  object 
 7   player_final_value  485409 non-null  object 
 8   actions_taken       485409 non-null  object 
 9   run_count           485409 non-null  int64  
 10  true_count          485409 non-null  int64  
 11  win                 485409 non-null  float64
dtypes: float64(1), int64(5), object(6)
memory usage: 48.1+ MB


## PREPROCESADO DEL DATASET

Si actions_taken hay mas de uno, mantener primera accion junto con mano inicial,
las siguientes acciones se añadiran como filas junto con el siguiente valor en la mano. Ej.:

Mano inicial [3, 9] | player_final [[3, 9, 4, 4]] | actions_taken ['H', 'H', 'S']

Filas resultantes:
 1- Mano inicial [3, 9] | actions_taken ['H']
 2- Mano inicial [3, 9, 4] | actions_taken ['H']
 3- Mano inicial [3, 9, 4, 4] | actions_taken ['S']


Los splits se tienen que separar tambien:

Mano inicial [3, 3] | player_final [[3, 2, 10], [3, 4, 5]] | actions_taken [['P', 'H', 'S'], ['H', 'S']]

Filas resultantes:
 1- Mano inicial [3, 3] | actions_taken ['P']
 2.1- Mano inicial [3, 2] | actions_taken ['H']
 2.2- Mano inicial [3, 2, 10] | actions_taken ['S']
 3.1- Mano inicial [3, 4] | actions_taken ['H']
 3.2- Mano inicial [3, 4, 5] | actions_taken ['S']

Columnas: 
- shoe_id: representa el taco de cartas que se esta usando, al dividir por acciones se usará el mismo, pero puede que exista tambien para otras partidas con otras acciones.
- cards_remaining, run_count y true_count hacen todas referencia a las cartas restantes y de momento se pueden usar las mismas para cada acción separada, si vemos que da problemas intentamos modificarlas
- dealer_up es la unica columna util del dealer, dealer_final y dealer_final_value las eliminamos
- player_final tambien sobra porque dividimos entre acciones.
- player_final_value se interpreta ahora como player_final_action_value, y es distinto para cada acción tomada. Para obtener player_final_action_value en caso de que haya un 11, comprobar si la suma se pasa de 21 y sumarlo como un 1 en caso de que si se pase.
- win se podría pasar a 0, 1 y 2 en caso de que queramos mantener el empate como posibilidad.


In [6]:
df["win"] = df["win"].apply(lambda x: 0 if x == 0 else 1 if x < 0 else 2) # 0 empate, 1 pierde, 2 gana

import re
import json
def str_to_list(cell, toInt = True):
    if "[[" in cell: cell = cell[1:-1]
    res = []
    for ls in cell.split("],"):
        ls = ''.join(c for c in ls if c not in "'[]").split(', ')
        
        if toInt: ls = list(map(int, ls))
        res.append(ls)

    if len(res) > 1:
        res = res
    else:
        res = res[0]
    
    return res

# if df["initial_hand"].dtype == str:
df["initial_hand"] = df["initial_hand"].apply(lambda x: str_to_list(x))
# if df["player_final"].dtype == str:
df["player_final"] = df["player_final"].apply(lambda x: str_to_list(x))
# if df["actions_taken"].dtype == str:
df["actions_taken"] = df["actions_taken"].apply(lambda x: str_to_list(x, False))

In [7]:
def create_row(df_row, hand, action = ""):
    res_row = []

    # Añadimos primera acción con initial_hand
    res_row.append(df_row[0]) # shoe_id
    res_row.append(df_row[1]) # cards_remaining
    res_row.append(df_row[2]) # dealer_up

    res_row.append(hand.copy()) # initial_hand -> player_hand
    if (11 in hand and sum(hand) > 21):
        for i in range(0, len(hand)):
            if hand[i] == 11 and sum(hand) > 21:
                hand[i] = 1
    res_row.append(sum(hand)) # player_final_value -> player_final_action_value

    if action == "":
        if type(df_row[6][0]) == str:
            res_row.append(df_row[6][0]) # actions_taken, first_action -> action
        else:
            res_row.append(df_row[6][0][0])
    else:
        res_row.append(action)

    res_row.append(df_row[7]) # run_count
    res_row.append(df_row[8]) # true_count
    res_row.append(df_row[9]) # win

    return res_row

In [8]:
original_cols = ["shoe_id", "cards_remaining", "dealer_up", "initial_hand", "player_final", "player_final_value", "actions_taken", "run_count", "true_count", "win"]
#                   0     |         1        |      2     |       3       |       4       |          5          |        6       |      7     |       8     |   9
df_list = df[original_cols].values.tolist()

new_df_list = []
for row in df_list:
    
    # Añadimos primera acción con initial_hand
    initial_row = create_row(row, row[3])
    new_df_list.append(initial_row)

    aux_row = initial_row
    if (type(row[4][0]) == int): # si no se ha hecho split
        for h, a in zip(row[4][2:], row[6][1:]):
            if aux_row[5] != "D":
                new_row = create_row(row, aux_row[3] + [h], a)
                aux_row = new_row

                new_df_list.append(new_row)
    else:
        if len(row[4]) <= 2: # Solo nos quedamos filas con un solo split
            for i, (split_hand, split_action) in enumerate(zip(row[4], row[6])): # split_hand = [8, 7, 4], split_action = ['P', 'H', 'S']
                if split_action[0] != "D":
                    # Primera mano del split
                    if i == 0:
                        initial_split_row = create_row(row, split_hand[:2], split_action[1])
                        new_df_list.append(initial_split_row)

                        aux_row_split = initial_split_row # aux_row_split = [683730, 320, 10, [8, 7], 15, 'H', -9, -1, -2.0]
                        for h, a in zip(split_hand[2:], split_action[2:]):
                            if aux_row_split[5] != "D":
                                new_split_row = create_row(row, aux_row_split[3] + [h], a)
                                aux_row_split = new_split_row

                                new_df_list.append(new_split_row)
                    else:
                        initial_split_row = create_row(row, split_hand[:2], split_action[0])
                        new_df_list.append(initial_split_row)

                        aux_row_split = initial_split_row # aux_row_split = [683730, 320, 10, [8, 7], 15, 'H', -9, -1, -2.0]
                        for h, a in zip(split_hand[2:], split_action[1:]):
                            if aux_row_split[5] != "D":
                                new_split_row = create_row(row, aux_row_split[3] + [h], a)
                                aux_row_split = new_split_row

                                new_df_list.append(new_split_row)


In [9]:
df_clean = pd.DataFrame(new_df_list)
df_clean.columns =["shoe_id", "cards_remaining", "dealer_up", "player_hand", "player_final_action_value", "action", "run_count", "true_count", "win"]
df_clean

,shoe_id,cards_remaining,dealer_up,player_hand,player_final_action_value,action,run_count,true_count,win
0,749782,79,4,"[8, 10]",18,S,-6,-3,2
1,401890,281,9,"[10, 9]",19,S,-11,-2,0
2,582076,398,9,"[10, 11]",21,S,2,0,2
3,3982,299,9,"[7, 8]",15,H,12,2,1
4,358698,259,8,"[10, 4]",14,H,2,0,1
...,...,...,...,...,...,...,...,...,...
694347,752255,174,10,"[6, 11]",17,H,14,4,2
694348,752255,174,10,"[6, 11, 5]",12,H,14,4,2
694349,752255,174,10,"[6, 11, 5, 9]",21,S,14,4,2
694350,212262,142,6,"[6, 2]",8,H,1,0,2


In [10]:
df_clean.to_csv("../BlackjackData/blackjack_dataset_clean.csv")